# VT_Publication_Figure

Builds the 12-panel Variable Tone figure (SHAM vs SNI) from the analysis pickle. No analysis is re-run; it runs in seconds.

**Pipeline:** 1. VT_Preprocessing → 2. VT_Publication_Analysis → **3. VT_Publication_Figure**

### Inputs (read from this repo)
| Path | Contents |
|---|---|
| `Photometry/VariableTone/RWD/RWD_Processed/vt_pub_data.pkl` | Output of VT_Publication_Analysis |
| `Photometry/VariableTone/vt_schematic.png` | Task schematic for panel A (optional; a placeholder is drawn if absent) |

### Outputs (downloaded at the end)
`VT_Publication_Figure.svg` and `.png`, and `VT_Publication_Figure_stats.csv` (every test in the figure).

### Statistics
Every test uses one value per mouse (n = mice). Two groups: Welch's t-test. Group x tone duration (or day): mixed ANOVA, then Welch's t per level, Holm-adjusted. Each panel shows the Holm-adjusted p and Hedges' g. The CSV also lists the 95% CI of g, the power to detect the observed effect, the smallest effect detectable with 80% power, and the mice per group needed for 80% power.

### Panels
- **Row 1:** A schematic · B retrieval latency · C distance from feeder at tone end (5 / 10 / 20 s) · D–F heatmap and mean trace per tone duration
- **Row 2:** G cue PSTH · H retrieval PSTH · I dispense PSTH · J peak dopamine at dispense by tone duration · K SHAM dispense kernels · L SNI dispense kernels

In [ ]:
# @title Setup — imports
import io
import re
import pickle
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.lines as mlines
import matplotlib.colors as _mc
from concurrent.futures import ThreadPoolExecutor, as_completed
from scipy.stats import ttest_ind
from scipy.ndimage import gaussian_filter1d as _gf1d
from tqdm.notebook import tqdm

print("Imports OK.")


Imports OK.


In [ ]:
# @title Source URLs & output path
# Inputs are read from the GitHub repo; outputs are written to the Colab runtime
# and downloaded at the end. Commit the outputs to the repo yourself.
import os

GITHUB_OWNER = "CreedLab-Neuro"   # or "MasBarr" if you point at a personal fork
GITHUB_REPO  = "Manuscript_PMID41279070"
GH_API = f"https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}"
VT     = "Photometry/VariableTone"

# ── Pin this run to one commit ────────────────────────────────────────────────
# raw.githubusercontent caches BRANCH urls (~5 min), so fetching ".../main/..."
# shortly after a push can serve the PREVIOUS pickle -- and you would not notice,
# because an old pickle plots perfectly happily. Resolve main to its commit SHA
# and fetch by SHA: those urls are immutable, so a cache hit is always correct.
DATA_SHA = requests.get(f"{GH_API}/commits/main").json()['sha']
GH_RAW   = f"https://raw.githubusercontent.com/{GITHUB_OWNER}/{GITHUB_REPO}/{DATA_SHA}"

PKL_URL       = f"{GH_RAW}/{VT}/RWD/RWD_Processed/vt_pub_data.pkl"   # all the figure plots
SCHEMATIC_URL = f"{GH_RAW}/{VT}/vt_schematic.png"     # panel A; optional
print(f"Data version (main @ commit): {DATA_SHA[:10]}")

OUTPUT_DIR = '/content/VT_Figures'                    # Colab-local; downloaded at the end
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Pickle    : {PKL_URL}\nSchematic : {SCHEMATIC_URL}\nOutput    : {OUTPUT_DIR}")


In [ ]:
# @title Load the pickle from GitHub — the only input this notebook has
_pr = requests.get(PKL_URL)
if _pr.status_code != 200:
    raise SystemExit(
        f"[STOP] could not fetch {PKL_URL} (HTTP {_pr.status_code}).\n"
        f"       Run VT_Publication_Analysis and commit vt_pub_data.pkl to the repo first.")
D = pickle.loads(_pr.content)

# ── Analysis outputs ──────────────────────────────────────────────────────────
vt_df               = D['vt_df']                # per-animal x per-tone metrics
vt_df_combined      = D['vt_df_combined']       # per-animal, tones pooled
event_time          = D['event_time']           # x-axis for the G/H/I PSTHs
psth_animals        = D['psth_animals']         # {event: {group: [per-animal traces]}}
disp_tone_animals   = D['disp_tone_animals']    # {group: {dur: [per-animal traces]}}
kern_by_group       = D['kern_by_group']        # pooled FIR kernels
kern_by_group_split = D['kern_by_group_split']  # tone-split FIR kernels (panels K/L)
kern_by_animal      = D['kern_by_animal']
t_ax                = D['t_ax']                 # x-axis for the kernels
GROUP_COLORS        = D['GROUP_COLORS']
PRES_FS             = D['PRES_FS']

# ── Per-trial figure inputs (precomputed in the analysis notebook) ────────────
_missing = [k for k in ('trial_psths', 'trial_dist_tone', 'animal_dist_tone', 'hm_time')
            if k not in D]
if _missing:
    raise SystemExit(
        f"[STOP] pickle is missing {_missing}.\n"
        f"       Re-run VT_Publication_Analysis and commit the new pickle.")

trial_psths      = D['trial_psths']       # {dur: {group: [(animal, trace)]}} -- heatmaps D/E/F
trial_dist_tone  = D['trial_dist_tone']   # {dur: {group: [float]}}           -- panel C dots
animal_dist_tone = D['animal_dist_tone']  # {dur: {animal: {group, mean}}}    -- panel C means
hm_time          = D['hm_time']           # x-axis for trial_psths traces

print("Loaded pickle. Keys:", list(D.keys()))


Loaded pickle. Keys: ['vt_df', 'vt_df_combined', 'event_time', 'psth_animals', 'disp_tone_animals', 'kern_by_group', 'kern_by_group_split', 'kern_by_animal', 't_ax', 'GROUP_COLORS', 'PRES_FS', 'trial_psths', 'trial_dist_tone', 'animal_dist_tone', 'hm_time']


In [ ]:
# @title Constants, publication style & helpers

# Publication colors — override whatever was saved in the pickle
GROUP_COLORS['SHAM'] = '#2166AC'   # blue
GROUP_COLORS['SNI']  = '#C0392B'   # red

PUB_GROUPS = ['SHAM', 'SNI']
DUR_STR    = ['5', '10', '20']
DUR_INT    = [5, 10, 20]
DUR_LS     = {'5': '-', '10': '--', '20': ':'}
LW_MEAN    = 3.0
LW_TONE    = 1.8
ALPHA_TONE = 0.60

# Larger publication fonts
PRES_FS = {'title': 21, 'label': 18, 'tick': 16, 'legend': 16}

plt.rcParams.update({
    'axes.linewidth': 2.5,
    'xtick.major.width': 2.5,
    'ytick.major.width': 2.5,
    'xtick.major.size':  6,
    'ytick.major.size':  6,
    'font.size': 17,
})

def mean_sem(M):
    gm = np.nanmean(M, axis=0)
    gs = np.nanstd(M, axis=0, ddof=1) / np.sqrt(M.shape[0])
    return gm, gs

def despine(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

def sig_bracket(ax, x1, x2, y, h, text, fontsize=10):
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], lw=1.2, color='k')
    ax.text((x1 + x2) / 2, y + h * 1.1, text, ha='center', va='bottom',
            fontsize=fontsize)

print("Style + helpers ready.")


Style + helpers ready.


In [ ]:
# @title Statistics helpers — animal-level tests, effect size, power
# Every test in this figure is run on ONE value per mouse (n = mice, not trials).
#   Two groups, one value per mouse  -> Welch's t-test
#   Two groups x tone duration       -> mixed ANOVA (group between, duration within),
#                                       then Welch's t per duration, Holm-adjusted
#   Effect size  -> Hedges' g on mouse means (positive = second group higher), 95% CI
#   Power        -> power to detect the observed g with these n; the smallest g these n
#                   detect with 80% power; mice per group needed for 80% power at the
#                   observed g (two-sided alpha = 0.05)
# The primary test of each outcome is Holm-adjusted across all outcomes in the figure.
import sys, subprocess
try:
    import pingouin as pg
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pingouin'], check=True)
    import pingouin as pg
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.power import TTestIndPower

ALPHA, TARGET_POWER = 0.05, 0.80
_POWER = TTestIndPower()

def hedges_g(a, b):
    """Hedges' g = (mean b - mean a) / pooled SD, small-sample corrected, with 95% CI."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    n1, n2 = len(a), len(b)
    sp = np.sqrt(((n1 - 1) * a.var(ddof=1) + (n2 - 1) * b.var(ddof=1)) / (n1 + n2 - 2))
    g  = (b.mean() - a.mean()) / sp * (1 - 3 / (4 * (n1 + n2) - 9))
    se = np.sqrt((n1 + n2) / (n1 * n2) + g ** 2 / (2 * (n1 + n2)))
    return g, g - 1.96 * se, g + 1.96 * se

def power_row(g, n1, n2):
    """Observed power, minimum detectable g (80%), and mice/group needed for 80%."""
    r = n2 / n1
    mde = _POWER.solve_power(nobs1=n1, ratio=r, alpha=ALPHA, power=TARGET_POWER)
    if abs(g) < 0.05:
        return dict(power_obs=ALPHA, mde_g=mde, n_per_group_80=np.inf)
    pw  = _POWER.power(effect_size=abs(g), nobs1=n1, ratio=r, alpha=ALPHA)
    need = _POWER.solve_power(effect_size=abs(g), ratio=1.0, alpha=ALPHA, power=TARGET_POWER)
    return dict(power_obs=pw, mde_g=mde, n_per_group_80=int(np.ceil(need)))

def welch_row(a, b):
    """Welch's t on mouse values + effect size + power."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    t = ttest_ind(a, b, equal_var=False)
    g, lo, hi = hedges_g(a, b)
    return dict(n1=len(a), n2=len(b), mean1=a.mean(), mean2=b.mean(), sd1=a.std(ddof=1),
                sd2=b.std(ddof=1), t=t.statistic, df=getattr(t, 'df', np.nan), p=t.pvalue,
                g=g, g_lo=lo, g_hi=hi, **power_row(g, len(a), len(b)))

def mixed_anova_row(long, dv, g1, g2):
    """Mixed ANOVA (group x duration) on a long table with columns animal, group, dur, dv.
    Only mice with all durations enter (a requirement of the ANOVA). Effect size and power
    use each mouse's mean over durations, which is what the group main effect tests."""
    d = long[long['group'].isin([g1, g2])].dropna(subset=[dv])
    d = d.groupby('animal').filter(lambda x: x['dur'].nunique() == d['dur'].nunique())
    aov = pg.mixed_anova(data=d, dv=dv, within='dur', between='group', subject='animal')
    pcol = 'p-unc' if 'p-unc' in aov.columns else 'p_unc'
    G, I = aov[aov.Source == 'group'].iloc[0], aov[aov.Source == 'Interaction'].iloc[0]
    m = d.groupby(['animal', 'group'])[dv].mean().reset_index()
    a, b = m[m.group == g1][dv].values, m[m.group == g2][dv].values
    g, lo, hi = hedges_g(a, b)
    return dict(n1=len(a), n2=len(b), mean1=a.mean(), mean2=b.mean(), sd1=a.std(ddof=1),
                sd2=b.std(ddof=1), F=G['F'], df=f"{int(G['DF1'])},{int(G['DF2'])}", p=G[pcol],
                np2=G['np2'], p_interaction=I[pcol], g=g, g_lo=lo, g_hi=hi,
                **power_row(g, len(a), len(b)))

def holm(pvals):
    p = np.asarray(pvals, float); out = np.full_like(p, np.nan); ok = np.isfinite(p)
    if ok.any(): out[ok] = multipletests(p[ok], method='holm')[1]
    return out

def fmt_p(p):
    if not np.isfinite(p): return 'p = n/a'
    return 'p < 0.001' if p < 0.001 else f'p = {p:.3f}' if p < 0.01 else f'p = {p:.2f}'

def p_g(row, adj=True):
    """'p = 0.21, g = 0.60' for a panel annotation (Holm-adjusted p when available)."""
    p = row.get('p_holm', row['p']) if adj else row['p']
    return f"{fmt_p(p)}, g = {row['g']:.2f}"

print("Stats helpers ready.")


In [ ]:
# @title Per-trial data — read straight from the pickle
# These used to be rebuilt here by iterating all 24 merged CSVs. They are now
# precomputed in VT_Publication_Analysis and travel in the pickle, so this
# notebook needs no CSVs at all. The panel code below is unchanged; these
# assignments just map the pickle keys onto the names it already uses.

_trial_pstths     = trial_psths        # {dur: {group: [(animal, trace)]}}  -- D/E/F heatmaps
_trial_dist_tone  = trial_dist_tone    # {dur: {group: [float]}}            -- C per-trial dots
_animal_dist_tone = animal_dist_tone   # {dur: {animal: {group, mean}}}     -- C per-animal dots

# Recover the window constants from hm_time so the panel code's
# np.linspace(-_PRE_HM, _WIN_END, _N_PTS) reproduces hm_time exactly.
_PRE_HM  = float(-hm_time[0])
_WIN_END = float(hm_time[-1])
_N_PTS   = len(hm_time)
t_win    = hm_time

print("Per-trial heatmap rows:",
      {d: {g: len(_trial_pstths[d][g]) for g in PUB_GROUPS} for d in ['5','10','20']})
print(f"Heatmap window: {_PRE_HM:.1f}s pre -> {_WIN_END:.1f}s post, {_N_PTS} points")


Per-trial heatmap rows: {'5': {'SHAM': 65, 'SNI': 62}, '10': {'SHAM': 50, 'SNI': 47}, '20': {'SHAM': 55, 'SNI': 53}}
Heatmap window: 1.0s pre -> 24.0s post, 300 points


In [ ]:
# @title Statistics — every test in the figure (printed and saved as a CSV)
# One value per mouse enters every test. Panel B: each mouse's mean retrieval latency.
# Panel C: each mouse's mean distance from the FED at tone end, per tone duration.
# Panel J: each mouse's peak DA after dispense, per tone duration.
STAT_CONTRASTS = [('SHAM', 'SNI')]   # (group 1, group 2); g > 0 means group 2 is higher
FIG_NAME       = 'VT_Publication_Figure'

_B = vt_df_combined[['animal', 'group', 'ret_latency']].dropna()
_C = pd.DataFrame([dict(animal=a, group=d['group'], dur=int(k), dist=d['mean'])
                   for k, ad in animal_dist_tone.items() for a, d in ad.items()]).dropna()
_J = vt_df.rename(columns={'tone_dur': 'dur'})[['animal', 'group', 'dur', 'peak_da']].dropna()
_OUTCOMES = [('C', 'Distance from FED at tone end (cm)', _C, 'dist'),
             ('J', 'Peak DA after dispense (z)',         _J, 'peak_da')]

_rows = []
for g1, g2 in STAT_CONTRASTS:
    con = f'{g1} vs {g2}'
    _rows.append(dict(panel='B', outcome='Retrieval latency (s)', contrast=con, dur='all',
                      level='primary', test='Welch t',
                      **welch_row(_B[_B.group == g1].ret_latency, _B[_B.group == g2].ret_latency)))
    for panel, outcome, long, dv in _OUTCOMES:
        _rows.append(dict(panel=panel, outcome=outcome, contrast=con, dur='all', level='primary',
                          test='Mixed ANOVA, group effect (group x duration)',
                          **mixed_anova_row(long, dv, g1, g2)))
        post = []
        for dur in DUR_INT:
            s = long[long.dur == dur]
            post.append(dict(panel=panel, outcome=outcome, contrast=con, dur=dur,
                             level='per duration', test='Welch t',
                             **welch_row(s[s.group == g1][dv], s[s.group == g2][dv])))
        for r, p in zip(post, holm([r['p'] for r in post])):   # Holm across the 3 durations
            r['p_holm'] = p
        _rows += post

STATS = pd.DataFrame(_rows)
_prim = STATS.level == 'primary'
STATS.loc[_prim, 'p_holm'] = holm(STATS.loc[_prim, 'p'])   # Holm across the figure's outcomes

def stat_row(panel, g1, g2, dur='all'):
    m = (STATS.panel == panel) & (STATS.contrast == f'{g1} vs {g2}') & (STATS.dur.astype(str) == str(dur))
    return STATS[m].iloc[0].to_dict()

def group_effect_text(panel):
    """One line per contrast: the primary test of this panel."""
    out = []
    for g1, g2 in STAT_CONTRASTS:
        r = stat_row(panel, g1, g2)
        stat = f"F({r['df']}) = {r['F']:.2f}" if np.isfinite(r.get('F', np.nan)) else f"t = {r['t']:.2f}"
        out.append(f"{g1} vs {g2}: {stat}, {fmt_p(r['p_holm'])}, g = {r['g']:.2f}")
    return '\n'.join(out)

def animal_means(rows):
    """[(animal, trace), ...] -> one mean trace per mouse (rows of the returned array)."""
    by = {}
    for a, tr in rows:
        by.setdefault(a, []).append(np.asarray(tr, float))
    return np.vstack([np.nanmean(np.vstack(v), axis=0) for v in by.values()])

pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30)
_show = ['panel', 'outcome', 'contrast', 'dur', 'test', 'n1', 'n2', 'mean1', 'mean2', 'p', 'p_holm',
         'p_interaction', 'np2', 'g', 'g_lo', 'g_hi', 'power_obs', 'mde_g', 'n_per_group_80']
print(STATS[[c for c in _show if c in STATS]].round(3).to_string(index=False))
out_csv = f"{OUTPUT_DIR}/{FIG_NAME}_stats.csv"
STATS.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")


In [ ]:
# @title Build the 12-panel figure & save (SVG + PNG)

fig = plt.figure(figsize=(26, 12))

# Row 1 and Row 2 use independent GridSpecs (different column counts)
gs_r1 = gridspec.GridSpec(
    1, 6, figure=fig,
    left=0.04, right=0.955, top=0.96, bottom=0.56,
    width_ratios=[1.1, 0.45, 1.4, 1.1, 1.1, 1.1],
    wspace=0.40,
)
gs_r2 = gridspec.GridSpec(
    1, 6, figure=fig,
    left=0.04, right=0.99, top=0.44, bottom=0.06,
    width_ratios=[1.2, 1.2, 1.2, 0.9, 1.2, 1.2],
    wspace=0.40,
)

axd = {}

axd['A'] = fig.add_subplot(gs_r1[0, 0])
axd['B'] = fig.add_subplot(gs_r1[0, 1])
gs_c = gridspec.GridSpecFromSubplotSpec(1, 3, subplot_spec=gs_r1[0, 2], wspace=0.55)
axd['C5']  = fig.add_subplot(gs_c[0])
axd['C10'] = fig.add_subplot(gs_c[1])
axd['C20'] = fig.add_subplot(gs_c[2])
for col_idx, dur in zip([3, 4, 5], ['5', '10', '20']):
    gs_def = gridspec.GridSpecFromSubplotSpec(
        2, 1, subplot_spec=gs_r1[0, col_idx],
        hspace=0.08, height_ratios=[2.2, 1],
    )
    axd[f'HM{dur}'] = fig.add_subplot(gs_def[0])
    axd[f'LP{dur}'] = fig.add_subplot(gs_def[1])

axd['G'] = fig.add_subplot(gs_r2[0, 0])
axd['H'] = fig.add_subplot(gs_r2[0, 1])
axd['I'] = fig.add_subplot(gs_r2[0, 2])
axd['J'] = fig.add_subplot(gs_r2[0, 3])
axd['K'] = fig.add_subplot(gs_r2[0, 4])
axd['L'] = fig.add_subplot(gs_r2[0, 5])

_PANEL_LABELS = {
    'A': 'A', 'B': 'B', 'C5': 'C',
    'HM5': 'D', 'HM10': 'E', 'HM20': 'F',
    'G': 'G', 'H': 'H', 'I': 'I', 'J': 'J', 'K': 'K', 'L': 'L',
}
for key, lbl in _PANEL_LABELS.items():
    axd[key].text(-0.14, 1.07, lbl, transform=axd[key].transAxes,
                  fontsize=16, fontweight='bold', va='top')

# ── A: Task schematic image ───────────────────────────────────────────────────
for sp in axd['A'].spines.values(): sp.set_visible(False)
axd['A'].set_xticks([]); axd['A'].set_yticks([])
try:
    _ir = requests.get(SCHEMATIC_URL); _ir.raise_for_status()
    _img = plt.imread(io.BytesIO(_ir.content), format='png')
    axd['A'].imshow(_img, aspect='equal')
except Exception as _e:
    print(f"[panel A] no schematic loaded from {SCHEMATIC_URL}  ({type(_e).__name__}: {_e})"
          f"\n          commit a PNG there and re-run this cell to fill panel A.")
    axd['A'].text(0.5, 0.5, 'Task Schematic\n(insert illustration)',
                  ha='center', va='center', fontsize=10, color='lightgray',
                  transform=axd['A'].transAxes)

# ── B / C: Behavioral plots ───────────────────────────────────────────────────
df_pub = vt_df_combined[vt_df_combined['group'].isin(PUB_GROUPS)].copy()

def pub_box(ax, data_by_grp, animal_means, ylabel, sig_data=None, show_trials=True):
    """Box-and-whisker with per-trial hollow dots and per-animal solid dots."""
    bp_data = [data_by_grp[grp] for grp in PUB_GROUPS]
    bp = ax.boxplot(bp_data, positions=[0, 1], widths=0.45, patch_artist=True,
                    medianprops=dict(color='k', lw=2),
                    whiskerprops=dict(lw=1.2), capprops=dict(lw=1.2),
                    flierprops=dict(marker=''))
    for patch, grp in zip(bp['boxes'], PUB_GROUPS):
        patch.set_facecolor(GROUP_COLORS[grp])
        patch.set_alpha(0.55)
    if show_trials:
        for xi, grp in enumerate(PUB_GROUPS):
            vals = np.array(data_by_grp[grp])
            jit  = np.random.default_rng(xi + 42).uniform(-0.15, 0.15, len(vals))
            ax.scatter(xi + jit, vals, s=14, color=GROUP_COLORS[grp],
                       edgecolors=GROUP_COLORS[grp], facecolors='none',
                       linewidths=0.8, alpha=0.55, zorder=3)
    if animal_means is not None:
        for xi, grp in enumerate(PUB_GROUPS):
            ameans = [d['mean'] for d in animal_means.values() if d['group'] == grp]
            jit    = np.random.default_rng(xi + 77).uniform(-0.10, 0.10, len(ameans))
            ax.scatter(xi + jit, ameans, s=40, color=GROUP_COLORS[grp],
                       edgecolors='k', linewidths=0.6, zorder=5)
    if sig_data:
        # sig_data: [(x1, x2, text), ...] -- one bracket per tested contrast
        _all = np.concatenate([np.asarray(data_by_grp[g], float) for g in PUB_GROUPS])
        yt   = np.nanmax(_all)
        sp2  = (yt - np.nanmin(_all)) * 0.12
        for k, (x1, x2, txt) in enumerate(sig_data):
            sig_bracket(ax, x1, x2, yt + sp2 * (0.6 + 2.6 * k), sp2 * 0.4, txt, fontsize=10)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(PUB_GROUPS, fontsize=PRES_FS['tick'])
    ax.set_ylabel(ylabel, fontsize=PRES_FS['label'])
    ax.tick_params(labelsize=PRES_FS['tick'])
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True, nbins=3))
    despine(ax)

# Panel B: ret_latency — per-animal box + per-animal dots (no trial-level)
_ret_by_grp = {grp: df_pub[df_pub['group'] == grp]['ret_latency'].dropna().tolist()
               for grp in PUB_GROUPS}
_ret_animal  = {row['animal']: {'group': row['group'], 'mean': row['ret_latency']}
                for _, row in df_pub.iterrows() if pd.notna(row['ret_latency'])}
pub_box(axd['B'], _ret_by_grp, _ret_animal, 'Retrieval Latency (s)',
        sig_data=[(PUB_GROUPS.index(a), PUB_GROUPS.index(b), p_g(stat_row('B', a, b)).replace(', ', '\n'))
                  for a, b in STAT_CONTRASTS], show_trials=False)

# Panel C: dist from FED at tone end, split by tone duration
for dur_str, ax_key in [('5','C5'),('10','C10'),('20','C20')]:
    ax = axd[ax_key]
    td = _trial_dist_tone[dur_str]
    ad = _animal_dist_tone[dur_str]
    # boxes summarise mouse means (n = mice); faint open dots are single trials
    bp_data = [[d['mean'] for d in ad.values() if d['group'] == g] for g in PUB_GROUPS]
    bp = ax.boxplot(bp_data, positions=[0,1], widths=0.45, patch_artist=True,
                    medianprops=dict(color='k', lw=1.5),
                    whiskerprops=dict(lw=1.0), capprops=dict(lw=1.0),
                    flierprops=dict(marker=''))
    for patch, grp in zip(bp['boxes'], PUB_GROUPS):
        patch.set_facecolor(GROUP_COLORS[grp]); patch.set_alpha(0.55)
    for xi, grp in enumerate(PUB_GROUPS):
        vals = np.array(td[grp])
        jit  = np.random.default_rng(xi+42).uniform(-0.14, 0.14, len(vals))
        ax.scatter(xi+jit, vals, s=8, color=GROUP_COLORS[grp],
                   facecolors='none', edgecolors=GROUP_COLORS[grp],
                   linewidths=0.6, alpha=0.45, zorder=3)
    for xi, grp in enumerate(PUB_GROUPS):
        ameans = [d['mean'] for d in ad.values() if d['group']==grp]
        jit    = np.random.default_rng(xi+77).uniform(-0.09, 0.09, len(ameans))
        ax.scatter(xi+jit, ameans, s=25, color=GROUP_COLORS[grp],
                   edgecolors='k', linewidths=0.5, zorder=5)
    # Per-duration test on mouse means, Holm-adjusted across the 3 durations
    all_t = np.concatenate([np.asarray(td[g], float) for g in PUB_GROUPS])
    yt    = np.nanmax(all_t)
    sp    = (yt - np.nanmin(all_t)) * 0.10
    for k, (g1, g2) in enumerate(STAT_CONTRASTS):
        _r = stat_row('C', g1, g2, int(dur_str))
        sig_bracket(ax, PUB_GROUPS.index(g1), PUB_GROUPS.index(g2), yt + sp * (0.5 + 3.2 * k),
                    sp * 0.4, f"{fmt_p(_r['p_holm'])}\ng = {_r['g']:.2f}", fontsize=9)
    ax.set_xticks([0,1])
    ax.set_xticklabels(PUB_GROUPS)
    ax.set_title(f'{dur_str}s', fontsize=PRES_FS['title'], fontweight='bold')
    ax.tick_params(labelsize=PRES_FS['tick'])
    # C5/C10/C20 are three narrow panels sharing the width of one normal panel,
    # so SHAM/SNI collide at the standard tick size. Must come AFTER the
    # tick_params above, which would otherwise reset both axes' label size.
    ax.tick_params(axis='x', labelsize=PRES_FS['tick']-3)
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True, nbins=3))
    despine(ax)
    if ax_key == 'C5':
        ax.set_ylabel('Dist from FED (cm)', fontsize=PRES_FS['label'])
_c_lo = min(axd[k].get_ylim()[0] for k in ['C5','C10','C20'])
_c_hi = max(axd[k].get_ylim()[1] for k in ['C5','C10','C20'])
for k in ['C5','C10','C20']:
    axd[k].set_ylim(_c_lo, _c_hi)
    axd[k].set_ylim(_c_lo, _c_hi + (_c_hi - _c_lo) * 0.18 * len(STAT_CONTRASTS))   # room for the p/g labels
axd['C10'].text(0.5, -0.13, group_effect_text('C'), transform=axd['C10'].transAxes,
                ha='center', va='top', fontsize=10)

# ── D / E / F: Heatmap + line plot per tone duration ─────────────────────────
def plot_hm_lp(dur_str, show_ylabel=False):
    ax_hm = axd[f'HM{dur_str}']
    ax_lp = axd[f'LP{dur_str}']
    dur_s = int(dur_str)
    t_win = np.linspace(-_PRE_HM, _WIN_END, _N_PTS)
    trials = _trial_pstths[dur_str]

    tone_mask = (t_win >= 0) & (t_win <= dur_s)
    def _peak_z(row): return np.nanmax(row[1][tone_mask]) if tone_mask.any() else 0
    sham_rows = sorted(trials['SHAM'], key=_peak_z, reverse=True)
    sni_rows  = sorted(trials['SNI'],  key=_peak_z, reverse=True)
    n_sham, n_sni = len(sham_rows), len(sni_rows)
    sep = np.full((2, _N_PTS), np.nan)
    parts = []
    if sham_rows: parts.append(np.array([r[1] for r in sham_rows]))
    parts.append(sep)
    if sni_rows:  parts.append(np.array([r[1] for r in sni_rows]))
    mat = np.vstack(parts)

    ax_hm.imshow(mat, aspect='auto', interpolation='nearest',
                 cmap='magma', vmin=0, vmax=20,
                 extent=[t_win[0], t_win[-1], mat.shape[0] - 0.5, -0.5],
                 origin='upper')

    ax_hm.axvline(0,     color='white', lw=3.0, ls=':', alpha=0.95)
    ax_hm.axvline(dur_s, color='gold',  lw=3.0, ls=':', alpha=0.95)
    for sp in ax_hm.spines.values(): sp.set_visible(False)

    ax_hm.set_yticks([n_sham / 2, n_sham + 2 + n_sni / 2])
    ax_hm.set_yticklabels(['SHAM', 'SNI'], fontsize=11, rotation=90, va='center')
    ax_hm.tick_params(axis='y', length=0)
    for lbl, grp in zip(ax_hm.get_yticklabels(), ['SHAM', 'SNI']):
        lbl.set_color(GROUP_COLORS[grp])
        lbl.set_fontweight('bold')
    ax_hm.set_xticks([])

    for grp in PUB_GROUPS:
        if not trials[grp]: continue
        M = animal_means(trials[grp])          # one row per mouse
        gm, gs_sem = mean_sem(M)               # SEM across mice (n = mice)
        ax_lp.plot(t_win, gm, color=GROUP_COLORS[grp], lw=LW_MEAN)
        ax_lp.fill_between(t_win, gm - gs_sem, gm + gs_sem,
                           color=GROUP_COLORS[grp], alpha=0.18)
    ax_lp.axvline(0,     color='gray', lw=2.5, ls=':', alpha=0.8)
    ax_lp.axvline(dur_s, color='gray', lw=2.5, ls=':', alpha=0.8)
    ax_lp.axhline(0,     color='gray', lw=0.6, ls=':', alpha=0.4)
    ax_lp.set_xlim(-_PRE_HM, _WIN_END)
    ax_lp.set_xticks([0, 5, 10, 15, 20])
    ax_lp.set_ylim(0, 24)
    ax_lp.set_yticks([0, 5, 10, 15, 20])
    ax_lp.set_xlabel('Time from tone onset (s)', fontsize=PRES_FS['label'])
    ax_lp.tick_params(labelsize=PRES_FS['tick'])
    if show_ylabel:
        ax_lp.set_ylabel('Z-score', fontsize=PRES_FS['label'])
    ax_lp.text(0.97, 0.95, f'{dur_str}s Tone', transform=ax_lp.transAxes,
               ha='right', va='top', fontsize=PRES_FS['legend'],
               fontweight='bold', color='#333333')
    despine(ax_lp)

plot_hm_lp('5',  show_ylabel=True)
plot_hm_lp('10')
plot_hm_lp('20')

# Colorbar to the right of panel F (HM20)
_im20    = axd['HM20'].images[0]
_pos     = axd['HM20'].get_position()
_cbar_ax = fig.add_axes([_pos.x1 + 0.005, _pos.y0, 0.010, _pos.height])
_cb      = fig.colorbar(_im20, cax=_cbar_ax)
_cb.set_ticks([0, 5, 10, 15, 20])
_cb.set_label('Z-score', fontsize=PRES_FS['label'] - 2)
_cb.ax.tick_params(labelsize=PRES_FS['tick'] - 2)

# ── G / H / I: Full-experiment Cue, Retrieval, Dispense PSTHs ────────────────
def plot_psth(ax, event_key, title, xlim=(-2, 4), show_ylabel=False, show_legend=False):
    for grp in PUB_GROUPS:
        traces = psth_animals[event_key][grp]
        if not traces: continue
        M = np.vstack(traces)
        gm, gs = mean_sem(M)
        ax.plot(event_time, gm, color=GROUP_COLORS[grp], lw=LW_MEAN, label=grp)
        ax.fill_between(event_time, gm - gs, gm + gs, color=GROUP_COLORS[grp], alpha=0.18)
    ax.axvline(0, color='gray', lw=0.8, ls='--', alpha=0.6)
    ax.axhline(0, color='gray', lw=0.5, ls=':', alpha=0.4)
    ax.set_xlim(*xlim)
    ax.set_title(title, fontsize=PRES_FS['title'], fontweight='bold')
    ax.set_xlabel('Time (s)', fontsize=PRES_FS['label'])
    if show_ylabel:
        ax.set_ylabel('Z-score', fontsize=PRES_FS['label'])
    if show_legend:
        ax.legend(frameon=False, fontsize=PRES_FS['legend'])
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True, nbins=5))
    ax.tick_params(labelsize=PRES_FS['tick'])
    despine(ax)

plot_psth(axd['G'], 'Cue',       'Cue',       xlim=(-2, 4), show_ylabel=True, show_legend=True)
plot_psth(axd['H'], 'Retrieval', 'Retrieval', xlim=(-2, 2))
plot_psth(axd['I'], 'Dispense',  'Dispense',  xlim=(-2, 4))

# ── J: Peak DA box-and-whisker by tone duration ───────────────────────────────
vt_pub = vt_df[vt_df['group'].isin(PUB_GROUPS)].copy()
bw   = 0.30
goff = {'SHAM': -bw / 2 - 0.02, 'SNI': bw / 2 + 0.02}
xmap = {5: 0, 10: 1, 20: 2}

for grp in PUB_GROUPS:
    for td in DUR_INT:
        vals = vt_pub[(vt_pub['group'] == grp) & (vt_pub['tone_dur'] == td)]['peak_da'].dropna().values
        xi   = xmap[td] + goff[grp]
        bp   = axd['J'].boxplot([vals], positions=[xi], widths=bw * 0.85,
                                patch_artist=True,
                                medianprops=dict(color='k', lw=2),
                                whiskerprops=dict(lw=1.2), capprops=dict(lw=1.2),
                                flierprops=dict(marker=''))
        bp['boxes'][0].set_facecolor(GROUP_COLORS[grp])
        bp['boxes'][0].set_alpha(0.55)
        jit = np.random.default_rng(td * 7 + PUB_GROUPS.index(grp)).uniform(-0.07, 0.07, len(vals))
        axd['J'].scatter(xi + jit, vals, s=25, color=GROUP_COLORS[grp],
                         edgecolors='k', linewidths=0.5, zorder=5)

ymax_J = vt_pub['peak_da'].dropna().max()
span_J = (ymax_J - vt_pub['peak_da'].dropna().min()) * 0.15
# Per-duration test on mouse values, Holm-adjusted across the 3 durations
for td in DUR_INT:
    _top = vt_pub[vt_pub['tone_dur'] == td]['peak_da'].max()
    for k, (g1, g2) in enumerate(STAT_CONTRASTS):
        _r = stat_row('J', g1, g2, td)
        sig_bracket(axd['J'], xmap[td] + goff[g1], xmap[td] + goff[g2],
                    _top + span_J * (0.4 + 1.6 * k), span_J * 0.25,
                    f"{fmt_p(_r['p_holm'])}\ng = {_r['g']:.2f}", fontsize=9)
_jlo, _jhi = axd['J'].get_ylim()
axd['J'].set_ylim(_jlo, _jhi + span_J * 0.8 * len(STAT_CONTRASTS))   # room for the p/g labels
axd['J'].text(0.5, -0.17, group_effect_text('J'), transform=axd['J'].transAxes,
              ha='center', va='top', fontsize=10)

axd['J'].set_xticks([0, 1, 2])
axd['J'].set_xticklabels(['5s', '10s', '20s'], fontsize=PRES_FS['tick'])
axd['J'].set_ylabel('Peak Z-score', fontsize=PRES_FS['label'])
axd['J'].set_xlabel('Tone Duration', fontsize=PRES_FS['label'])
axd['J'].tick_params(labelsize=PRES_FS['tick'])
axd['J'].yaxis.set_major_locator(plt.MaxNLocator(integer=True, nbins=3))
despine(axd['J'])

# ── K / L: Dispense kernel split by tone duration — SHAM / SNI ───────────────
def _shade(hex_color, lightness_factor):
    """Blend hex_color toward white; factor 1.0 = original, 0.3 = very light."""
    r, g, b = _mc.to_rgb(hex_color)
    r = r + (1 - r) * (1 - lightness_factor)
    g = g + (1 - g) * (1 - lightness_factor)
    b = b + (1 - b) * (1 - lightness_factor)
    return (r, g, b)

_KERN_SMOOTH_SIGMA = 1   # 1 sample ~ 67 ms at 15 Hz
_DISP_DURS    = ['5', '10', '20']
_SHADE_LEVELS = {'5': 0.35, '10': 0.65, '20': 1.0}   # lighter -> darker

def plot_disp_kern(ax, grp, title, show_ylabel=False):
    base_col = GROUP_COLORS[grp]
    all_curves = []
    for dur in _DISP_DURS:
        ks = kern_by_group_split[grp].get(f'Dispense {dur}', [])
        if not ks:
            continue
        M = np.vstack(ks)
        if len(M) == 0:
            continue
        gm = _gf1d(np.nanmean(M, axis=0), sigma=_KERN_SMOOTH_SIGMA)
        col = _shade(base_col, _SHADE_LEVELS[dur])
        ax.plot(t_ax, gm, color=col, lw=1.8, label=f'{dur}s')
        all_curves.append(gm)
    if all_curves:
        avg = _gf1d(np.nanmean(np.vstack(all_curves), axis=0), sigma=_KERN_SMOOTH_SIGMA)
        ax.plot(t_ax, avg, color='black', lw=2.5, ls='-', label='avg')
    ax.axvline(0, color='gray', lw=0.8, ls='--', alpha=0.6)
    ax.axhline(0, color='gray', lw=0.5, ls=':', alpha=0.4)
    ax.set_xlim(-2, 4)
    ax.set_title(title, fontsize=PRES_FS['title'], fontweight='bold')
    ax.set_xlabel('Time (s)', fontsize=PRES_FS['label'])
    if show_ylabel:
        ax.set_ylabel('Z-score', fontsize=PRES_FS['label'])
    ax.legend(frameon=False, fontsize=PRES_FS['legend'])
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True, nbins=5))
    ax.tick_params(labelsize=PRES_FS['tick'])
    despine(ax)

# Titles kept short: the full 'Dispense Kernel — SHAM' is wide enough that the
# centred title runs under the bold panel letter at x=-0.14.
plot_disp_kern(axd['K'], 'SHAM', 'SHAM Kernel', show_ylabel=True)
plot_disp_kern(axd['L'], 'SNI',  'SNI Kernel')

# Sync K/L y-axes
_kl_lo = min(axd['K'].get_ylim()[0], axd['L'].get_ylim()[0])
_kl_hi = max(axd['K'].get_ylim()[1], axd['L'].get_ylim()[1])
for _ax in [axd['K'], axd['L']]:
    _ax.set_ylim(_kl_lo, _kl_hi)
    _ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True, nbins=3))

# Sync G/H/I y-axes (PSTH panels)
_ghi_lo = min(axd[k].get_ylim()[0] for k in ['G', 'H', 'I'])
_ghi_hi = max(axd[k].get_ylim()[1] for k in ['G', 'H', 'I'])
for _ax in [axd['G'], axd['H'], axd['I']]:
    _ax.set_ylim(_ghi_lo, _ghi_hi)
    _ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True, nbins=3))

# ── Statistics footnote ───────────────────────────────────────────────────────
_nB = ', '.join(f"{g} {int(((_B.group == g)).sum())}" for g in PUB_GROUPS)
_nC = ', '.join(f"{g} {_C[_C.group == g].animal.nunique()}" for g in PUB_GROUPS)
_mde = stat_row('B', *STAT_CONTRASTS[0])
fig.text(0.04, -0.06,
         f"Statistics on mouse means (n = mice: {_nB}; panel C {_nC}). B: Welch's t-test. "
         f"C, J: group x tone-duration mixed ANOVA, then "
         f"Welch's t per duration, Holm-adjusted across durations. Group-effect p values (text under C and J) are "
         f"Holm-adjusted across B, C and J. g = Hedges' g (second group minus first).\n"
         f"Power: with n = {int(_mde['n1'])} vs {int(_mde['n2'])} mice, a two-sided test at "
         f"alpha = 0.05 has 80% power only for g >= {_mde['mde_g']:.2f}. Full table: {FIG_NAME}_stats.csv.",
         ha='left', va='top', fontsize=12, color='#333333')

# ── Save (SVG + PNG, Colab-local) and show ────────────────────────────────────
out_svg = f"{OUTPUT_DIR}/VT_Publication_Figure.svg"
out_png = f"{OUTPUT_DIR}/VT_Publication_Figure.png"
plt.savefig(out_svg, format='svg', bbox_inches='tight')
plt.savefig(out_png, dpi=150, bbox_inches='tight')
print(f"Saved: {out_svg}")
print(f"Saved: {out_png}")
plt.show()


In [ ]:
# @title Download the figure (SVG + PNG)
# Colab-local output does not survive the runtime. The SVG is the one to keep --
# it is vector, so it stays sharp at any journal's figure width.
from google.colab import files

for _f in (out_svg, out_png, out_csv):
    print(f"{_f}  ({os.path.getsize(_f)/1e6:.2f} MB)")
    files.download(_f)
